In [1]:
import os, glob
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm
sys.path.append("..")
from shared_util.time_handler import TimeHandler

In [2]:
# 与源目录结构保持一致，争对aiops代码
'''
datasets/
└── train/
    └── 2022-03-21/
        └── cloudbed-1/
            └── metric/
                └── container/
                    └── kpi_container_cpu_cfs_periods.csv
                └── istio/
                    └── kpi_istio_agent_endpoint_no_pod.csv                
                └── jvm/
                    └── kpi_istio_agent_endpoint_no_pod.csv                               
                └── node/
                    └── kpi_cloudbed1_metric_0320.csv                
                └── service/
                    └── metric_service.csv


针对四种不同层级的不同实体，分别得到了一张横表
timestamp metric1 metric2 ...
'''

'\ndatasets/\n└── train/\n    └── 2022-03-21/\n        └── cloudbed-1/\n            └── metric/\n                └── container/\n                    └── kpi_container_cpu_cfs_periods.csv\n                └── istio/\n                    └── kpi_istio_agent_endpoint_no_pod.csv                \n                └── jvm/\n                    └── kpi_istio_agent_endpoint_no_pod.csv                               \n                └── node/\n                    └── kpi_cloudbed1_metric_0320.csv                \n                └── service/\n                    └── metric_service.csv\n'

In [2]:



# =========================
# 1) 路径解析：dataset_type/date/cloudbed（用切割/relpath，和你 log 一致）
# =========================
def parse_aiops_ids_from_metric_root(metric_root: str, data_base: str):
    """
    metric_root: {data_base}/{dataset_type}/{date}/{cloudbed}/metric
    """
    rel = os.path.relpath(metric_root, data_base)
    parts = rel.split(os.sep)
    if len(parts) < 4 or parts[-1] != "metric":
        raise ValueError(f"Unexpected metric path: {rel}")
    return parts[0], parts[1], parts[2]


# =========================
# 2) 时间桶：复刻（24h，每分钟）
# =========================
def build_day_minute_timestamps(date: str, TimeHandler):
    start_ts = TimeHandler.datetime_to_timestamp(f"{date} 00:00:00")
    return [start_ts + i * 60 for i in range(0, 24 * 60)]


def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)
    return p


# =========================
# 3) 自动扫描：推断有哪些 metric 文件 / 列
# =========================
def list_csv_files(folder: str):
    return sorted(glob.glob(os.path.join(folder, "*.csv"))) if os.path.isdir(folder) else []


def infer_container_metric_files(metric_root: str):
    """
    container 每个指标一个 csv：container/<raw_metric_name>.csv
    返回 raw_metric_name 列表（去掉 .csv）
    """
    files = list_csv_files(os.path.join(metric_root, "container"))
    return [os.path.splitext(os.path.basename(p))[0] for p in files]


def infer_istio_metric_files(metric_root: str):
    files = list_csv_files(os.path.join(metric_root, "istio"))
    return [os.path.splitext(os.path.basename(p))[0] for p in files]


def infer_node_kpi_names(metric_root: str):
    """
    node 通常一个大表，kpi_name 在行里：
      timestamp, cmdb_id, kpi_name, value
    我们直接读取一次拿 unique kpi_name
    """
    files = list_csv_files(os.path.join(metric_root, "node"))
    print(files)
    if not files:
        return None, []
    df = pd.read_csv(files[0], usecols=["kpi_name"])
    return files[0], sorted(df["kpi_name"].dropna().astype(str).unique().tolist())


def infer_service_columns(metric_root: str):
    """
    service 通常一个大表：
      timestamp, service, <many metric columns...>
    我们读取表头推断有哪些列可用
    """
    files = list_csv_files(os.path.join(metric_root, "service"))
    if not files:
        return None, []
    df_head = pd.read_csv(files[0], nrows=1)
    cols = [c for c in df_head.columns if c not in ("timestamp", "service")]
    return files[0], cols


# =========================
# 4) node metric：按 (timestamp, node, kpi_name) 精确取 value
# 输出表格的格式  
'''
    timestamp  CPU/system.load.1  CPU/system.load.15  CPU/system.load.5  \  
    CPU/system.cpu.iowait  CPU/system.cpu.pct_usage  CPU/system.cpu.system  \
    CPU/system.cpu.user  Memory/system.mem.pct_usage  \
    Memory/system.mem.real.pct_useage  ...  Disk/system.io.r_await  \
    Disk/system.io.r_s  Disk/system.io.rkb_s  Disk/system.io.svctm  \
    Disk/system.io.util  Disk/system.io.w_await  Disk/system.io.w_s  \
    Disk/system.disk.free  Disk/system.disk.pct_usage  Disk/system.disk.used
'''
# =========================
def process_node_metric_auto(
    timestamp_list: list,
    node_file: str,
    out_root: str,
    node_list: list,
    node_metric_name_dict,
):
    '''
        node_metric_dict[timestamp][metric_name]——>df(timestamp, node, raw_metric_name)
            metric_name = f'{metric_type}/{raw_metric_name}'
                metric_type: CPU Memory Disk
                raw_metric_name: system.load.1
    '''

    metric_df = pd.read_csv(node_file)
    '''
        timestamp: s为单位
        cmdb_id: node-1                   
        kpi_name: system.load.1
        value: 1.04
    '''
    metric_df.set_index(["timestamp", "cmdb_id", "kpi_name"], inplace=True, drop=False)
    out_dir = ensure_dir(os.path.join(out_root, "node"))
    for node in node_list:
        node_metric_dict = {"timestamp": timestamp_list}

        for metric_type in node_metric_name_dict:
            metric_name_list = node_metric_name_dict[metric_type]
            for raw_metric_name in metric_name_list:
                # 格式为类似于 CPU/system.load.1
                metric_name = f'{metric_type}/{raw_metric_name}'
                node_metric_dict[metric_name] = []
                for timestamp in timestamp_list:
                    # 这里没有按照时间桶，因为这里的指标已经是分钟对齐的
                    if (timestamp, node, raw_metric_name) in metric_df.index:
                        node_metric_dict[metric_name].append(metric_df.loc[timestamp, node, raw_metric_name]['value'])
                    else:
                        node_metric_dict[metric_name].append(np.nan)
        # 得到的横表为timestamp | CPU/system.load.1 | CPU/system.load.15 | CPU/system.cpu.user | ...
        node_df = pd.DataFrame(node_metric_dict)
        node_df.to_csv(os.path.join(out_dir, f"{node}.csv"), index=False)


# =========================
# 5) container metric：每个指标文件一列；按 contains(pod) 找唯一匹配
# =========================
def process_container_metric_auto(
    timestamp_list: list,
    metric_root: str,
    out_root: str,
    pod_list: list,
    metric_info_dict,
):
    '''
        container_metric_dict[pod] = {
            'timestamp': timestamp_list,
            '{metric_type}/{raw_metric_name}': df
        }
        分别存储每个pod的横表
    '''

    out_dir = ensure_dir(os.path.join(out_root, "container"))
    T = len(timestamp_list)
    # pod -> dict
    container_metric_dict = {pod: {"timestamp": timestamp_list} for pod in pod_list}

    for metric_type in metric_info_dict:
        metric_name_list = metric_info_dict[metric_type]
        metric_name_bar = tqdm(metric_name_list)
        for raw_metric_name in metric_name_bar:
            metric_name = f'{metric_type}/{raw_metric_name}'

            metric_df = pd.read_csv(os.path.join(metric_root, "container", f"{raw_metric_name}.csv"))
            '''
                timestamp: s为单位
                cmdb_id: node-5.checkoutservice2-0          
                kpi_name: container_cpu_usage_seconds
                value: 1.04
            '''

            metric_df.set_index(['timestamp', 'cmdb_id'], inplace=True, drop=False)

            for pod in pod_list:
                container_metric_dict[pod][metric_name] = []
                for timestamp in timestamp_list:
                    is_find = False
                    if timestamp in metric_df.index.levels[0]:
                        temp_df = metric_df.loc[timestamp]
                        temp_df = temp_df[temp_df.cmdb_id.str.contains(pod)]
                        if temp_df.shape[0] == 1:
                            container_metric_dict[pod][metric_name].append(temp_df.iloc[0]['value'])
                            is_find = True
                    if not is_find:
                        container_metric_dict[pod][metric_name].append(np.nan)
            metric_name_bar.set_description("Container metric csv generating".format(metric_name))

    for pod, metric_dict in container_metric_dict.items():
        metric_df = pd.DataFrame(metric_dict)
        metric_df.to_csv(os.path.join(out_dir, f"{pod}.csv"), index=False)
# =========================
# 6) istio metric：每个指标文件 -> 两列 source/destination；多行 sum
# =========================
def process_istio_metric_auto(
    timestamp_list: list,
    metric_root: str,
    out_root: str,
    pod_list: list,
    istio_metric_name_dict,
):

    out_dir = ensure_dir(os.path.join(out_root, "istio"))
    T = len(timestamp_list)

    istio_metric_dict = {pod: {"timestamp": timestamp_list} for pod in pod_list}
    '''
        istio_metric_dict[pod] = {
            'timestamp': timestamp_list,
            '{metric_type}/{raw_metric_name}.source': df
            '{metric_type}/{raw_metric_name}.destination': df
        }
        分别存储每个pod的横表
    '''


    for metric_type, metric_name_list in istio_metric_name_dict.items():
        metric_name_bar = tqdm(metric_name_list)
        for raw_metric_name in metric_name_bar:
            fpath = os.path.join(metric_root, "istio", f"{raw_metric_name}.csv")
            
            metric_name = f'{metric_type}/{raw_metric_name}'
            metric_df = pd.read_csv(fpath)
            '''
                timestamp: s为单位
                cmdb_id: adservice2-0.destination.frontend2.adservice2
                kpi_name: istio_request_bytes.grpc.200.9.0
                value: 1.04
            '''
            metric_df.set_index(['timestamp', 'cmdb_id', 'kpi_name'], inplace=True, drop=False)

            for pod in pod_list:
                istio_metric_dict[pod][f'{metric_name}.source'] = []
                istio_metric_dict[pod][f'{metric_name}.destination'] = []
                for timestamp in timestamp_list:
                    if timestamp in metric_df.index.levels[0]:
                        temp_df = metric_df.loc[timestamp]
                        source_temp_df = temp_df[temp_df.cmdb_id.str.contains(f'{pod}.source')]

                        destination_temp_df = temp_df[temp_df.cmdb_id.str.contains(f'{pod}.destination')]

                        if source_temp_df.shape[0] > 0:
                            istio_metric_dict[pod][f'{metric_name}.source'].append(source_temp_df['value'].sum())
                        else:
                            istio_metric_dict[pod][f'{metric_name}.source'].append(np.nan)

                        if destination_temp_df.shape[0] > 0:
                            istio_metric_dict[pod][f'{metric_name}.destination'].append(
                                destination_temp_df['value'].sum())
                        else:
                            istio_metric_dict[pod][f'{metric_name}.destination'].append(np.nan)
                    else:
                        istio_metric_dict[pod][f'{metric_name}.source'].append(np.nan)
                        istio_metric_dict[pod][f'{metric_name}.destination'].append(np.nan)

            metric_name_bar.set_description("Istio metric csv generating".format(metric_name))

    for pod, metric_dict in istio_metric_dict.items():
        metric_df = pd.DataFrame(metric_dict)
        metric_df.to_csv(os.path.join(out_dir, f"{pod}.csv"), index=False)
# =========================
# 7) service metric：从 service/*.csv 的列名自动推断；输出 grpc/http 两列
# =========================
def process_service_metric_auto(
    timestamp_list: list,
    service_file: str,
    out_root: str,
    service_list: list,
    service_metric_name_dict,
):

    '''
        这里虽然service_metric_dict没有svc这一级，但是for循环是按照每个svc定义的，所以相当于还有一级
        service_metric_dict[svc] = {
            'timestamp': timestamp_list,
            '{metric_type}/{raw_metric_name}.grpc': df
            '{metric_type}/{raw_metric_name}.http': df
        }
        分别存储每个pod的横表
    '''
    metric_df = pd.read_csv(service_file)
    '''
        service   timestamp     rr          sr       mrt  count
        0   paymentservice-grpc  1651507200  100.0  100.000000  0.000000      0
        1        adservice-http  1651507200  100.0  100.000000  8.931250     24
        2        adservice-grpc  1651507200  100.0   75.221239  2.315487    113
        3  checkoutservice-grpc  1651507200  100.0  100.000000  0.000000      0
        4  shippingservice-grpc  1651507200  100.0  100.000000  0.000000      1
    '''
    metric_df.set_index(["timestamp", "service"], inplace=True, drop=False)

    out_dir = ensure_dir(os.path.join(out_root, "service"))

    service_list_bar = tqdm(service_list)
    for service in service_list_bar:
        service_metric_dict = {"timestamp": timestamp_list}

        for metric_type, metric_name_list in service_metric_name_dict.items():
            for raw_metric_name in metric_name_list:
                metric_name = f'{metric_type}/{raw_metric_name}'
                service_metric_dict[f'{metric_name}.grpc'] = []
                service_metric_dict[f'{metric_name}.http'] = []

                for timestamp in timestamp_list:
                    if timestamp in metric_df.index.levels[0]:
                        temp_df = metric_df.loc[timestamp]
                        temp_df_grpc = temp_df[temp_df.service == f'{service}-grpc']
                        if temp_df_grpc.shape[0] == 1:
                            service_metric_dict[f'{metric_name}.grpc'].append(temp_df_grpc.iloc[0][raw_metric_name])
                        else:
                            service_metric_dict[f'{metric_name}.grpc'].append(np.nan)

                        temp_df_http = temp_df[temp_df.service == f'{service}-http']
                        if temp_df_http.shape[0] == 1:
                            service_metric_dict[f'{metric_name}.http'].append(temp_df_http.iloc[0][raw_metric_name])
                        else:
                            service_metric_dict[f'{metric_name}.http'].append(np.nan)
                    else:
                        service_metric_dict[f'{metric_name}.grpc'].append(np.nan)
                        service_metric_dict[f'{metric_name}.http'].append(np.nan)
        service_df = pd.DataFrame(service_metric_dict)
        service_df.to_csv(os.path.join(out_dir, f"{service}.csv"), index=False)
        service_list_bar.set_description("Service metric csv generating".format(service))
# =========================
# 8) 总控：extract_metric_features（完全像你 log 一样：os.walk + skip + 镜像输出）
# =========================
def extract_metric_features_auto(
    data_base: str,
    result_base_path: str,
    TimeHandler,
    node_list: list,
    pod_list: list,
    service_list: list,
    node_metric_name_dict,
    container_metric_name_dict,
    istio_metric_name_dict,
    service_metric_name_dict,
    metric_folder_name="metric",
    skip_if_path_contains=("test_data",),
    skip_bad_case=True,
):
    for root, dirs, files in os.walk(data_base):
        if os.path.basename(root) != metric_folder_name:
            continue

        metric_root = root  # .../<dataset_type>/<date>/<cloudbed>/metric

        # if any(x in metric_root for x in skip_if_path_contains):
        #     continue

        dataset_type, date, cloudbed = parse_aiops_ids_from_metric_root(metric_root, data_base)

        if skip_bad_case and date == "2022-03-24" and cloudbed in ["cloudbed-1", "cloudbed-2"]:
            continue

        timestamp_list = build_day_minute_timestamps(date, TimeHandler)

        out_root = ensure_dir(os.path.join(result_base_path, dataset_type, date, cloudbed, "raw_metric"))

        print(f"[INFO] metric: {dataset_type} {date} {cloudbed}")

        # 1) 构建按时间戳对齐的node相关指标的横表
        node_file, node_kpis = infer_node_kpi_names(metric_root)
        process_node_metric_auto(timestamp_list, node_file, out_root, node_list, node_metric_name_dict)

        # 2) 构建按时间戳对齐的pod相关指标的横表
        container_metrics = infer_container_metric_files(metric_root)
        process_container_metric_auto(timestamp_list, metric_root, out_root, pod_list, container_metric_name_dict)

        # 3) 自动推断 istio 指标文件
        istio_metrics = infer_istio_metric_files(metric_root)
        process_istio_metric_auto(timestamp_list, metric_root, out_root, pod_list, istio_metric_name_dict)

        # 4) 自动推断 service 指标列
        service_file, service_cols = infer_service_columns(metric_root)
        process_service_metric_auto(timestamp_list, service_file, out_root, service_list, service_metric_name_dict)

In [3]:
pod_list = [
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [4]:
service_order_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

In [5]:
node_metric_name_dict = {
    'CPU': [
        'system.load.1', 
        'system.load.15', 
        'system.load.5', 
        'system.cpu.iowait', 
        'system.cpu.pct_usage', 
        'system.cpu.system', 
        'system.cpu.user'
    ], 
    'Memory': [
        'system.mem.pct_usage', 
        'system.mem.real.pct_useage', 
        'system.mem.real.used', 
        'system.mem.usable', 
        'system.mem.used'
    ], 
    'Disk': [
        'system.io.avg_q_sz', 
        'system.io.await', 
        'system.io.r_await', 
        'system.io.r_s', 
        'system.io.rkb_s', 
        'system.io.svctm', 
        'system.io.util', 
        'system.io.w_await', 
        'system.io.w_s', 
        'system.disk.free', 
        'system.disk.pct_usage', 
        'system.disk.used'
    ]
}

In [6]:
container_metric_name_dict={
    'CPU': [
        'kpi_container_cpu_usage_seconds', 
        'kpi_container_cpu_user_seconds', 
        'kpi_container_cpu_system_seconds', 
        'kpi_container_cpu_cfs_throttled_seconds', 
        'kpi_container_cpu_cfs_throttled_periods'
    ], 
    'Memory': [
        'kpi_container_memory_cache', 
        'kpi_container_memory_mapped_file', 
        'kpi_container_memory_usage_MB', 
        'kpi_container_memory_working_set_MB', 
        'kpi_container_memory_rss'
    ], 
    'Disk': [
        'kpi_container_fs_inodes', 
        'kpi_container_fs_reads_MB', 
        'kpi_container_fs_usage_MB', 
        'kpi_container_fs_writes_MB'
    ], 
    'Thread': [
        'kpi_container_threads'
    ], 
    'Network': [
        'kpi_container_network_receive_packets', 
        'kpi_container_network_receive_MB', 
        'kpi_container_network_transmit_packets', 
        'kpi_container_network_transmit_MB'
    ]
}

In [7]:
istio_metric_name_dict={
    'Network': [
        'kpi_istio_request_bytes', 
        'kpi_istio_request_duration_milliseconds', 
        'kpi_istio_request_messages', 
        'kpi_istio_requests', 
        'kpi_istio_response_bytes', 
        'kpi_istio_response_messages'
    ]
}

In [8]:
service_metric_name_dict={
    'Request': ['rr', 'sr', 'mrt', 'count']
}

In [9]:
node_list = ["node-1","node-2","node-3","node-4","node-5","node-6"]

extract_metric_features_auto(
    data_base="/home/ZZData/Eadro/preprocess/datasets/aiops2022/",
    result_base_path="../../temp_data/aiops22/raw_data",
    TimeHandler=TimeHandler,
    node_list=node_list,
    pod_list=pod_list,
    service_list=service_order_list,
    node_metric_name_dict=node_metric_name_dict,
    container_metric_name_dict = container_metric_name_dict,
    istio_metric_name_dict=istio_metric_name_dict,
    service_metric_name_dict=service_metric_name_dict,
    skip_if_path_contains=("test_data",),
)


[INFO] metric: test_data 2022-05-03 cloudbed
['/home/ZZData/Eadro/preprocess/datasets/aiops2022/test_data/2022-05-03/cloudbed/metric/node/kpi_cloudbed2_metric_0324.csv']


Service metric csv generating: 100%|██████████| 10/10 [00:21<00:00,  2.13s/it]


[INFO] metric: test_data 2022-05-01 cloudbed
['/home/ZZData/Eadro/preprocess/datasets/aiops2022/test_data/2022-05-01/cloudbed/metric/node/kpi_cloudbed1_metric_0324.csv']


Service metric csv generating: 100%|██████████| 10/10 [00:18<00:00,  1.89s/it]


[INFO] metric: test_data 2022-05-09 cloudbed
['/home/ZZData/Eadro/preprocess/datasets/aiops2022/test_data/2022-05-09/cloudbed/metric/node/kpi_cloudbed1_metric_0315.csv']


...

Service metric csv generating: 100%|██████████| 10/10 [00:23<00:00,  2.31s/it]


[INFO] metric: training_data_with_faults 2022-03-21 cloudbed-2
['/home/ZZData/Eadro/preprocess/datasets/aiops2022/training_data_with_faults/2022-03-21/cloudbed-2/metric/node/kpi_cloudbed2_metric_0321.csv']


Service metric csv generating: 100%|██████████| 10/10 [00:23<00:00,  2.30s/it]


[INFO] metric: training_data_with_faults 2022-03-24 cloudbed-3
['/home/ZZData/Eadro/preprocess/datasets/aiops2022/training_data_with_faults/2022-03-24/cloudbed-3/metric/node/kpi_cloudbed3_metric_0324.csv']


Service metric csv generating: 100%|██████████| 10/10 [00:23<00:00,  2.32s/it]
